# 03_Feature_Engineering — Rossmann Store Sales
Lanjutan dari `02_Data_Cleaning.ipynb`. Di sini kita:
- Menggabungkan (`merge`) `train_clean` dan `store_clean`, yang sengaja **tidak** dilakukan di tahap cleaning.
- Membuat fitur turunan (date features, competition features, promotion features, sales metrics) yang juga sengaja **tidak** dibuat di tahap cleaning.
- Menangani kolom yang sebelumnya dibiarkan missing (`CompetitionOpenSinceMonth/Year`, `Promo2SinceWeek/Year`, `PromoInterval`) sesuai konteks fitur yang dibuat di sini.

## 1. Import Libraries

In [20]:
import pandas as pd
import numpy as np
import os

## 2. Load Clean Dataset
Memuat hasil dari tahap cleaning (`cleaned/train_clean.csv` dan `cleaned/store_clean.csv`), bukan dari data mentah.

In [21]:
train_clean = pd.read_csv('cleaned/train_clean.csv', parse_dates=['Date'])
store_clean = pd.read_csv('cleaned/store_clean.csv')

print('train_clean shape:', train_clean.shape)
print('store_clean shape:', store_clean.shape)

train_clean shape: (1017209, 9)
store_clean shape: (1115, 10)


In [22]:
train_clean.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,NoHoliday,1
1,2,5,2015-07-31,6064,625,1,1,NoHoliday,1
2,3,5,2015-07-31,8314,821,1,1,NoHoliday,1
3,4,5,2015-07-31,13995,1498,1,1,NoHoliday,1
4,5,5,2015-07-31,4822,559,1,1,NoHoliday,1


In [23]:
store_clean.head()

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


## 3. Merge Dataset
Gabungkan `train_clean` dan `store_clean` berdasarkan kolom `Store`. Ini adalah titik pertama kedua dataset digabung — sesuai keputusan sebelumnya bahwa merge dilakukan di tahap Feature Engineering, bukan di tahap Cleaning.

In [24]:
missing_store_keys = set(train_clean['Store']) - set(store_clean['Store'])
print('Store IDs in train_clean but missing from store_clean:', missing_store_keys)

Store IDs in train_clean but missing from store_clean: set()


In [25]:
df = train_clean.merge(store_clean, on='Store', how='left')

print('Merged shape:', df.shape)
df.head()

Merged shape: (1017209, 18)


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-31,5263,555,1,1,NoHoliday,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,5,2015-07-31,6064,625,1,1,NoHoliday,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-07-31,8314,821,1,1,NoHoliday,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,5,2015-07-31,13995,1498,1,1,NoHoliday,1,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,5,2015-07-31,4822,559,1,1,NoHoliday,1,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


## 4. Create Date Features
Dibuat dari kolom `Date` yang sudah bertipe datetime sejak tahap cleaning.

In [26]:
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Quarter'] = df['Date'].dt.quarter
df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
df['IsWeekend'] = df['DayOfWeek'].isin([6, 7]).astype(int)

print(df[['Date', 'Year', 'Month', 'Quarter', 'Week', 'DayOfWeek', 'IsWeekend']].head(10))

        Date  Year  Month  Quarter  Week  DayOfWeek  IsWeekend
0 2015-07-31  2015      7        3    31          5          0
1 2015-07-31  2015      7        3    31          5          0
2 2015-07-31  2015      7        3    31          5          0
3 2015-07-31  2015      7        3    31          5          0
4 2015-07-31  2015      7        3    31          5          0
5 2015-07-31  2015      7        3    31          5          0
6 2015-07-31  2015      7        3    31          5          0
7 2015-07-31  2015      7        3    31          5          0
8 2015-07-31  2015      7        3    31          5          0
9 2015-07-31  2015      7        3    31          5          0


**Catatan:** `DayOfWeek` pada dataset Rossmann bernilai 1 (Senin) s.d. 7 (Minggu), sehingga `IsWeekend` diset `True` untuk nilai 6 (Sabtu) dan 7 (Minggu).

In [27]:
print('Year unique:', sorted(df['Year'].unique()))
print('Month range:', df['Month'].min(), '-', df['Month'].max())
print('Quarter unique:', sorted(df['Quarter'].unique()))
print('Week range:', df['Week'].min(), '-', df['Week'].max())
print('IsWeekend value counts:')
print(df['IsWeekend'].value_counts())

Year unique: [np.int32(2013), np.int32(2014), np.int32(2015)]
Month range: 1 - 12
Quarter unique: [np.int32(1), np.int32(2), np.int32(3), np.int32(4)]
Week range: 1 - 52
IsWeekend value counts:
IsWeekend
0    727749
1    289460
Name: count, dtype: int64


## 5. Create Competition Features
`CompetitionOpenSinceMonth` dan `CompetitionOpenSinceYear` sebelumnya sengaja **dibiarkan missing** di tahap cleaning. Di sini kita ubah menjadi satu fitur turunan: **`CompetitionDuration`** — lama (dalam bulan) kompetitor terdekat sudah beroperasi, dihitung relatif terhadap `Date` transaksi.

Untuk baris yang `CompetitionOpenSinceMonth`/`Year`-nya masih missing (kompetitor ada tapi tanggal buka tidak tercatat), `CompetitionDuration` dibiarkan `NaN` — bukan diisi 0, supaya tidak disalahartikan seolah kompetitor baru buka.

In [28]:
has_competition_date = df['CompetitionOpenSinceMonth'].notna() & df['CompetitionOpenSinceYear'].notna()

competition_duration_months = (
    (df['Year'] - df['CompetitionOpenSinceYear']) * 12
    + (df['Month'] - df['CompetitionOpenSinceMonth'])
)

df['CompetitionDuration'] = np.where(has_competition_date, competition_duration_months, np.nan)

# Kompetitor yang secara data 'buka' setelah tanggal transaksi -> durasi negatif.
# Ini secara logis berarti kompetitor belum beroperasi pada tanggal tsb, jadi di-clip ke 0.
df['CompetitionDuration'] = df['CompetitionDuration'].clip(lower=0)

print('CompetitionDuration missing:', df['CompetitionDuration'].isnull().sum())
print(df['CompetitionDuration'].describe())

CompetitionDuration missing: 323348
count    693861.000000
mean         61.631076
std          71.079571
min           0.000000
25%          14.000000
50%          51.000000
75%          94.000000
max        1386.000000
Name: CompetitionDuration, dtype: float64


## 6. Create Promotion Features
`Promo2SinceWeek`, `Promo2SinceYear`, dan `PromoInterval` sebelumnya sengaja **dibiarkan missing** di tahap cleaning karena bersifat *structural missing* (toko dengan `Promo2 = 0` memang tidak punya info ini). Di sini kita ubah menjadi **`PromoDuration`** — lama (dalam minggu) toko sudah mengikuti Promo2 berulang, relatif terhadap `Date` transaksi.

Untuk toko dengan `Promo2 = 0`, `PromoDuration` diisi **0** (bukan `NaN`) karena secara bisnis memang tidak ada durasi Promo2 yang berjalan — ini konsisten dengan makna 'tidak berlaku' yang sudah kita identifikasi sebelumnya.

In [29]:
# Konversi Year + Week menjadi perkiraan jumlah minggu sejak awal tahun
week_of_year = df['Date'].dt.isocalendar().week.astype(int)

promo_duration_weeks = (
    (df['Year'] - df['Promo2SinceYear']) * 52
    + (week_of_year - df['Promo2SinceWeek'])
)

df['PromoDuration'] = np.where(df['Promo2'] == 1, promo_duration_weeks, 0)

# Durasi negatif (Promo2 tercatat mulai setelah tanggal transaksi) di-clip ke 0
df['PromoDuration'] = df['PromoDuration'].clip(lower=0)

print('PromoDuration missing:', df['PromoDuration'].isnull().sum())
print(df.groupby('Promo2')['PromoDuration'].describe())

PromoDuration missing: 0
           count        mean        std  min   25%    50%    75%    max
Promo2                                                                 
0       508031.0    0.000000   0.000000  0.0   0.0    0.0    0.0    0.0
1       509178.0  113.432089  84.536284  0.0  36.0  108.0  181.0  312.0


## 7. Create Sales Metrics
**`SalesPerCustomer`** = `Sales / Customers`. Pada hari toko tutup (`Open = 0`), `Customers` bernilai 0, sehingga pembagian akan menghasilkan `NaN`/`inf`. Baris ini diberi nilai `0` karena memang tidak ada transaksi yang terjadi, bukan data yang perlu diimputasi.

In [30]:
df['SalesPerCustomer'] = np.where(
    df['Customers'] > 0,
    df['Sales'] / df['Customers'],
    0
)

print('SalesPerCustomer missing:', df['SalesPerCustomer'].isnull().sum())
print('SalesPerCustomer infinite values:', np.isinf(df['SalesPerCustomer']).sum())
print(df['SalesPerCustomer'].describe())

SalesPerCustomer missing: 0
SalesPerCustomer infinite values: 0
count    1.017209e+06
mean     7.880231e+00
std      4.089279e+00
min      0.000000e+00
25%      6.905375e+00
50%      8.683684e+00
75%      1.051098e+01
max      6.495785e+01
Name: SalesPerCustomer, dtype: float64


## 8. Validation
Validasi menyeluruh terhadap dataset akhir hasil feature engineering: struktur, tipe data, missing value, dan nilai tidak wajar (infinite / negatif) pada fitur baru.

In [31]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 26 columns):
 #   Column                     Non-Null Count    Dtype         
---  ------                     --------------    -----         
 0   Store                      1017209 non-null  int64         
 1   DayOfWeek                  1017209 non-null  int64         
 2   Date                       1017209 non-null  datetime64[us]
 3   Sales                      1017209 non-null  int64         
 4   Customers                  1017209 non-null  int64         
 5   Open                       1017209 non-null  int64         
 6   Promo                      1017209 non-null  int64         
 7   StateHoliday               1017209 non-null  str           
 8   SchoolHoliday              1017209 non-null  int64         
 9   StoreType                  1017209 non-null  str           
 10  Assortment                 1017209 non-null  str           
 11  CompetitionDistance        1017209 non-null  flo

In [32]:
print('Null values per column:')
print(df.isnull().sum())

Null values per column:
Store                             0
DayOfWeek                         0
Date                              0
Sales                             0
Customers                         0
Open                              0
Promo                             0
StateHoliday                      0
SchoolHoliday                     0
StoreType                         0
Assortment                        0
CompetitionDistance               0
CompetitionOpenSinceMonth    323348
CompetitionOpenSinceYear     323348
Promo2                            0
Promo2SinceWeek              508031
Promo2SinceYear              508031
PromoInterval                508031
Year                              0
Month                             0
Quarter                           0
Week                              0
IsWeekend                         0
CompetitionDuration          323348
PromoDuration                     0
SalesPerCustomer                  0
dtype: int64


In [33]:
new_features = ['Year', 'Month', 'Quarter', 'Week', 'IsWeekend',
                'SalesPerCustomer', 'CompetitionDuration', 'PromoDuration']

print('Summary statistik fitur baru:')
print(df[new_features].describe())

Summary statistik fitur baru:
               Year         Month       Quarter          Week     IsWeekend  \
count  1.017209e+06  1.017209e+06  1.017209e+06  1.017209e+06  1.017209e+06   
mean   2.013832e+03  5.846762e+00  2.294252e+00  2.361551e+01  2.845630e-01   
std    7.773960e-01  3.326097e+00  1.081850e+00  1.443338e+01  4.512063e-01   
min    2.013000e+03  1.000000e+00  1.000000e+00  1.000000e+00  0.000000e+00   
25%    2.013000e+03  3.000000e+00  1.000000e+00  1.100000e+01  0.000000e+00   
50%    2.014000e+03  6.000000e+00  2.000000e+00  2.200000e+01  0.000000e+00   
75%    2.014000e+03  8.000000e+00  3.000000e+00  3.500000e+01  1.000000e+00   
max    2.015000e+03  1.200000e+01  4.000000e+00  5.200000e+01  1.000000e+00   

       SalesPerCustomer  CompetitionDuration  PromoDuration  
count      1.017209e+06        693861.000000   1.017209e+06  
mean       7.880231e+00            61.631076   5.678000e+01  
std        4.089279e+00            71.079571   8.242528e+01  
min       

In [34]:
# Cek tidak ada nilai infinite pada kolom numerik hasil feature engineering
for col in ['SalesPerCustomer', 'CompetitionDuration', 'PromoDuration']:
    inf_count = np.isinf(df[col]).sum()
    print(f'{col}: infinite values = {inf_count}')

SalesPerCustomer: infinite values = 0
CompetitionDuration: infinite values = 0
PromoDuration: infinite values = 0


In [35]:
# Cek tidak ada nilai negatif pada durasi (sudah di-clip sebelumnya)
print('CompetitionDuration min:', df['CompetitionDuration'].min())
print('PromoDuration min:', df['PromoDuration'].min())

CompetitionDuration min: 0.0
PromoDuration min: 0.0


In [36]:
dup_count = df.duplicated().sum()
print('Duplicated rows after merge & feature engineering:', dup_count)

Duplicated rows after merge & feature engineering: 0


## 9. Export Processed Dataset
Disimpan ke folder terpisah `processed/`, supaya `raw/`, `raw_backup/`, dan `cleaned/` dari tahap-tahap sebelumnya tetap utuh.

In [37]:
processed_dir = 'processed'
os.makedirs(processed_dir, exist_ok=True)

df.to_csv(f'{processed_dir}/rossmann_processed.csv', index=False)

print('Processed file saved to:', processed_dir)
print(os.listdir(processed_dir))
print('Final shape:', df.shape)

Processed file saved to: processed
['rossmann_processed.csv']
Final shape: (1017209, 26)


## 10. Feature Engineering Summary

| Feature | Digunakan untuk Dashboard | Digunakan di SQL | Digunakan di Warehouse |
|---|---|---|---|
| Year | ✅ | ✅ | ✅ |
| Month | ✅ | ✅ | ✅ |
| Quarter | ✅ | ✅ | ✅ |
| Week | ✅ | ✅ | ✅ |
| IsWeekend | ✅ | ✅ | ✅ |
| SalesPerCustomer | ✅ | ✅ | ✅ |
| CompetitionDuration | ✅ | ✅ | ✅ |
| PromoDuration | ✅ | ✅ | ✅ |

**Ringkasan proses:**
- `train_clean` + `store_clean` digabung menjadi satu dataset (`df`) pada grain store-day.
- 5 fitur tanggal dibuat dari `Date`/`DayOfWeek`: `Year`, `Month`, `Quarter`, `Week`, `IsWeekend`.
- `CompetitionDuration` diturunkan dari `CompetitionOpenSinceMonth`/`Year`, dengan missing yang bersifat structural tetap dibiarkan `NaN` (bukan diisi 0).
- `PromoDuration` diturunkan dari `Promo2SinceWeek`/`Year`, dengan toko `Promo2 = 0` diberi nilai `0` (bukan `NaN`) karena secara bisnis memang tidak ada Promo2 yang berjalan.
- `SalesPerCustomer` diturunkan dari `Sales`/`Customers`, dengan hari toko tutup diberi nilai `0`.
- Dataset akhir divalidasi (tipe data, missing, infinite, duplikat) lalu diekspor ke `processed/rossmann_processed.csv`, siap untuk tahap Analyze / ETL / Load ke warehouse.